# Progress tracking and weighted selection

Demonstrates `UserWordProgress`, `compute_weight`, and `select_words`.


In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import random

from rich import print as rprint

from lang_tools.progress import SelectionWeights
from lang_tools.progress import UserWordProgress
from lang_tools.progress import WordFilter
from lang_tools.progress import compute_weight
from lang_tools.progress import select_words
from lang_tools.words.word import Word

## Build a small pool


In [ ]:
pool = [
    Word(text=t, language="pt", frequency=f, translations={"en": tr})
    for t, f, tr in [
        ("ação", "high", "action"),
        ("café", "high", "coffee"),
        ("mão", "medium", "hand"),
        ("livro", "medium", "book"),
        ("cadeira", "low", "chair"),
    ]
]
for w in pool:
    rprint(w.text, "id=", w.id)

## Track progress


In [ ]:
progress: dict[str, UserWordProgress] = {}
seen = pool[0]
p = UserWordProgress(user_id="alice", word_id=seen.id)
p.record(correct=False, exercise_type="wordle")
p.record(correct=True, exercise_type="wordle")
progress[seen.id] = p
rprint(p)

## Inspect weights

Unseen words get the unseen multiplier; seen-with-errors get an error boost.


In [ ]:
weights = SelectionWeights()
for w in pool:
    score = compute_weight(w, progress.get(w.id), weights)
    rprint(f"{w.text:>10}  weight={score:.2f}")

## Sample with a filter

Restrict to accented words with an English translation.


In [ ]:
chosen = select_words(
    pool,
    progress,
    n=3,
    word_filter=WordFilter(has_accent=True, has_translation="en"),
    weights=weights,
    rng=random.Random(0),
)
for w in chosen:
    rprint(w.text)